# Water Level Assembly and Shoreline Correction

## Imports and settings

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA   = os.path.join('..', 'data', 'water_levels')
TIDES  = os.path.join(DATA, 'QUINTA_tides.csv')        # dates, tide (15-min)
SLA    = os.path.join(DATA, 'sla_quinta_daily.csv')    # time, sla (daily)
RUNUP  = os.path.join(DATA, 'quinta_runup.csv')        # time + runup column(s)

RUNUP_COLS = None                  # e.g. ['R2'] or per-transect columns; None = all numeric columns
PERIOD     = ('1993-01-01', None)  # SLA starts 1993; (None, None) = whole tide record

OUT_TIDE_SLA       = os.path.join(DATA, 'QUINTA_tides_plus_sla.csv')
OUT_TIDE_SLA_RUNUP = os.path.join(DATA, 'QUINTA_tides_sla_runup.csv')

# shoreline correction (CoastSat, Vos et al. 2019)
COASTSAT      = os.path.join('..', 'data', 'coastsat')
SHORELINES    = os.path.join(COASTSAT, 'transect_time_series.csv')   # raw, NOT tidally corrected
SLOPE         = {
    'B6': 0.045,
    'C2': 0.040,
    'C1': 0.065,
    'B4': 0.040,
    'B5': 0.035,
    'A1': 0.055,
    'A2': 0.045,
    'B1': 0.040,
    'B2': 0.045,
    'B3': 0.045,
}      # foreshore slope per transect
DEFAULT_SLOPE = 0.08    # used for any transect not in SLOPE
REF_LEVEL     = 0.0     # m above MSL: contour all shorelines are moved to (CoastSat reference_elevation)
TOL           = '30min' # max time between an image and its water-level sample
META          = {'satname', 'geoaccuracy', 'cloud_cover', 'x', 'y', 'filename', 'idx'}

# Castelle et al. (2021) low-water threshold
TWL_THRESHOLD = 0.2     # m above MSL; keep only images with tide + SLA + runup above this (0.2 m at Truc Vert, site specific)

# despiking (CoastSat reject_outliers) and gap trimming
MAX_CROSS_CHANGE = 40                # m; max cross-shore change between consecutive points (CoastSat example: 40)
OTSU_THRESHOLD   = (-0.5, 0)         # keep images with MNDWI threshold in this range (CoastSat example); (np.nan, np.nan) = skip
OUTPUT_PKL       = os.path.join(COASTSAT, 'QUINTA_output.pkl')   # CoastSat output, only needed for OTSU_THRESHOLD
MAX_GAP          = '365D'            # each series starts after its last gap longer than this
COMMON_START     = False             # True = one start date for all transects (the latest one)

## Helper functions

In [ ]:
def load(path):
    """Read a CSV and index it by its time column (UTC, tz-naive)."""
    df = pd.read_csv(path)
    tcol = next((c for c in ['dates', 'date', 'time', 'datetime'] if c in df.columns), df.columns[0])
    df.index = pd.to_datetime(df[tcol], utc=True).dt.tz_localize(None)
    return df.drop(columns=[tcol]).sort_index()


def interp_to(s, target, max_gap):
    """Linear interpolation onto target times; NaN outside the record or across gaps > max_gap."""
    s = s.dropna()
    s = s[~s.index.duplicated()]
    x, xi = s.index.asi8.astype(float), target.asi8.astype(float)
    v = np.interp(xi, x, s.values, left=np.nan, right=np.nan)
    pos = np.searchsorted(x, xi).clip(1, len(x) - 1)
    v[(x[pos] - x[pos - 1]) > pd.Timedelta(max_gap).value] = np.nan
    return pd.Series(v, index=target)


def save(df, path):
    df.index.name = 'dates'
    df.round(4).to_csv(path)
    print(f'saved {os.path.basename(path)}: {len(df):,} rows, '
          f'{df.index[0]:%Y-%m-%d} to {df.index[-1]:%Y-%m-%d}')


def at_images(s, dates, tol=TOL):
    """Value at each image time: nearest sample within tol (CoastSat's get_closest_datapoint)."""
    s = s.dropna()
    s = s[~s.index.duplicated()]
    pos = s.index.get_indexer(dates, method='nearest', tolerance=pd.Timedelta(tol))
    return pd.Series(np.where(pos >= 0, s.values[pos.clip(0)], np.nan), index=dates)


def correct(raw, level, transects):
    """CoastSat correction: raw + (level - REF_LEVEL) / slope.
    level is one Series for all transects, or a DataFrame with one column per transect."""
    out = raw.copy()
    for t in transects:
        lev = level[t] if isinstance(level, pd.DataFrame) else level
        out[t] = raw[t] + (lev - REF_LEVEL) / SLOPE.get(t, DEFAULT_SLOPE)
    return out


def save_ts(df, name, transects):
    """Save in CoastSat's own CSV layout (dates as UTC, satname, one column per transect)."""
    path = os.path.join(COASTSAT, f'transect_time_series_{name}.csv')
    out = df.copy()
    out.index = out.index.tz_localize('UTC')
    out.rename_axis('dates').reset_index().to_csv(path, index=False)
    n = out[transects].notna().any(axis=1).sum()
    print(f'{name:14s} {n:4d} images with values -> {os.path.basename(path)}')

## Load data

Tides, daily SLA and runup are loaded once and shared by both outputs below.

In [ ]:
tide_all = load(TIDES)['tide']                  # full record, used for the shoreline correction
tide     = tide_all.loc[PERIOD[0]:PERIOD[1]]     # clipped, used for the water level CSVs
sla  = load(SLA)['sla']
run  = load(RUNUP)
run  = run[RUNUP_COLS] if RUNUP_COLS else run.select_dtypes('number')

print(f'tide  {tide_all.index[0]:%Y-%m-%d} to {tide_all.index[-1]:%Y-%m-%d}')
print(f'SLA   {sla.index[0]:%Y-%m-%d} to {sla.index[-1]:%Y-%m-%d}')
print(f'runup {run.index[0]:%Y-%m-%d} to {run.index[-1]:%Y-%m-%d}, columns {list(run.columns)}')

# seasonal cycle already in the tides (FES Sa/Ssa)? if so, remove it from the SLA to avoid counting it twice
tm = tide.resample('MS').mean()
p2p = tm.groupby(tm.index.month).mean().pipe(lambda s: s.max() - s.min())
if p2p > 0.02:
    clim = sla.groupby(sla.index.month).transform('mean')
    sla = sla - (clim - clim.mean())               # keeps the overall mean and trend
print(f'seasonal cycle in tides {100*p2p:.1f} cm -> SLA seasonal cycle {"removed" if p2p > 0.02 else "kept"}')

sla_i = interp_to(sla, tide.index, '3D')          # daily SLA onto the 15-min tide times

## Create water level CSVs

### Tide + SLA

In [ ]:
wl = pd.DataFrame({'tide': tide, 'sla': sla_i})
wl['water_level'] = wl['tide'] + wl['sla']        # NaN where SLA is missing
wl['sla_missing'] = wl['sla'].isna()

save(wl, OUT_TIDE_SLA)
print(f'SLA missing {wl.sla_missing.mean():.1%}')

### Tide + SLA + wave runup

In [ ]:
twl = pd.DataFrame({'tide': tide, 'sla': sla_i})
for c in run.columns:
    twl[f'runup_{c}'] = interp_to(run[c], tide.index, '3h')

twl['still_wl'] = twl['tide'] + twl['sla']                 # NaN where SLA is missing
for c in run.columns:
    twl[f'twl_{c}'] = twl['still_wl'] + twl[f'runup_{c}']  # NaN where any term is missing
twl['sla_missing']   = twl['sla'].isna()
twl['runup_missing'] = twl.filter(like='runup_').isna().any(axis=1)

save(twl, OUT_TIDE_SLA_RUNUP)
print(f'SLA missing {twl.sla_missing.mean():.1%}, runup missing {twl.runup_missing.mean():.1%}')
print(twl.filter(like='twl_').describe().loc[['mean', 'max']].round(2))

## Correction of Shorelines

Same correction as CoastSat (Vos et al., 2019): take the water level at each image time (nearest sample) and shift each chainage along its transect to a fixed reference elevation:

`corrected = raw + (water_level - REF_LEVEL) / beach_slope`

Only the water level changes between the four versions. Despike all four afterwards with the same CoastSat settings so they stay comparable.

In [ ]:
raw = load(SHORELINES)
raw = raw.loc[:, ~raw.columns.str.startswith('Unnamed')]
transects = [c for c in raw.columns if c not in META and pd.api.types.is_numeric_dtype(raw[c])]
dates = raw.index
print(f'{len(dates)} images, {dates[0]:%Y-%m-%d} to {dates[-1]:%Y-%m-%d}, transects {transects}')

### No Corrections

In [ ]:
ts_none = raw.copy()
save_ts(ts_none, 'none', transects)

### Tidal Correction

Uses the full tide record (`tide_all`), not just the 1993+ period.

In [ ]:
lev_tide = at_images(tide_all, dates)
ts_tide = correct(raw, lev_tide, transects)
save_ts(ts_tide, 'tide', transects)

### Tidal + SLA Correction

In [ ]:
lev_tide_sla = at_images(wl['water_level'], dates)
ts_tide_sla = correct(raw, lev_tide_sla, transects)
save_ts(ts_tide_sla, 'tide_sla', transects)

### Tidal + SLA + Wave Run Up

If the runup file has one column per transect, each transect uses its own; if it has a single column, that one is used for all transects.

In [ ]:
twl_cols = [c for c in twl.columns if c.startswith('twl_')]
if len(twl_cols) == 1:
    col_for = {t: twl_cols[0] for t in transects}
else:
    col_for = {t: next((c for c in twl_cols if c.endswith(t)), None) for t in transects}
    missing = [t for t, c in col_for.items() if c is None]
    assert not missing, f'no runup column found for transects {missing}'

cache = {c: at_images(twl[c], dates) for c in set(col_for.values())}
lev_wave = pd.DataFrame({t: cache[col_for[t]] for t in transects})
ts_tide_sla_wave = correct(raw, lev_wave, transects)
save_ts(ts_tide_sla_wave, 'tide_sla_wave', transects)

### Tidal + SLA + Wave Run Up + low-water threshold (Castelle et al., 2021)

Same correction as above, but images are kept only when the total water level at the image time is above `TWL_THRESHOLD`. Castelle et al. found 0.2 m optimal at Truc Vert and stress it is site specific, so treat it as a parameter to test against the drone surveys.

In [ ]:
keep = lev_wave.gt(TWL_THRESHOLD)                   # per transect, NaN water level counts as below
ts_tide_sla_wave_thr = ts_tide_sla_wave.copy()
for t in transects:
    ts_tide_sla_wave_thr.loc[~keep[t], t] = np.nan

n_before = ts_tide_sla_wave[transects].notna().any(axis=1).sum()
save_ts(ts_tide_sla_wave_thr, 'tide_sla_wave_thr', transects)
n_after = ts_tide_sla_wave_thr[transects].notna().any(axis=1).sum()
print(f'threshold {TWL_THRESHOLD} m removed {n_before - n_after} of {n_before} images')

### Summary

In [ ]:
levels = pd.DataFrame({'tide': lev_tide, 'tide_sla': lev_tide_sla,
                       'tide_sla_wave': lev_wave.mean(axis=1)})
levels.rename_axis('dates').to_csv(os.path.join(COASTSAT, 'water_levels_at_images.csv'))

common = levels.dropna().index                      # images every version could correct
print(f'{len(common)} of {len(dates)} images have all corrections\n')
t0 = transects[len(transects) // 2]
for name, df in [('none', ts_none), ('tide', ts_tide), ('tide_sla', ts_tide_sla), ('tide_sla_wave', ts_tide_sla_wave)]:
    s = df.loc[common, t0]
    print(f'{name:14s} {t0}: mean {s.mean():7.1f} m   std {s.std():5.1f} m')

## De-spiking and trimming

Each version is cleaned the same way, following CoastSat's `reject_outliers` (Vos et al., 2019):
1. drop NaNs,
2. drop images whose MNDWI (Otsu) threshold is outside `OTSU_THRESHOLD` (CoastSat drops the whole transect if fewer than 30 points remain),
3. iterative despiking with CoastSat's `identify_outliers` and `MAX_CROSS_CHANGE`,

then each series is trimmed to start after its last gap longer than `MAX_GAP`. Every removed point is logged with its reason.

In [ ]:
# ---- CoastSat identify_outliers (SDS_transects.py, unchanged) ----
def identify_outliers(chainage, dates, cross_change):
    chainage_temp = chainage.copy()
    dates_temp = dates.copy()
    k = 0
    while k < len(chainage_temp):
        for k in range(len(chainage_temp)):
            if k == 0:
                diff = chainage_temp[k] - chainage_temp[k+1]
                if np.abs(diff) > cross_change:
                    chainage_temp.pop(k); dates_temp.pop(k); break
            elif k == len(chainage_temp)-1:
                diff = chainage_temp[k] - chainage_temp[k-1]
                if np.abs(diff) > cross_change:
                    chainage_temp.pop(k); dates_temp.pop(k); break
            else:
                diff_m1 = chainage_temp[k] - chainage_temp[k-1]
                diff_p1 = chainage_temp[k] - chainage_temp[k+1]
                condition1 = np.abs(diff_m1) > cross_change
                condition2 = np.abs(diff_p1) > cross_change
                condition3 = np.sign(diff_p1) == np.sign(diff_m1)
                if np.logical_and(np.logical_and(condition1, condition2), condition3):
                    chainage_temp.pop(k); dates_temp.pop(k); break
                if k >= 2 and k < len(chainage_temp)-2:
                    diff_m2 = chainage_temp[k-1] - chainage_temp[k-2]
                    diff_p2 = chainage_temp[k+1] - chainage_temp[k+2]
                    condition4 = np.abs(diff_m2) > cross_change
                    condition5 = np.abs(diff_p2) > cross_change
                    condition6 = np.sign(diff_m1) == np.sign(diff_p2)
                    condition7 = np.sign(diff_p1) == np.sign(diff_m2)
                    if np.logical_and(np.logical_and(condition1, condition5), condition6):
                        chainage_temp.pop(k); dates_temp.pop(k); break
                    elif np.logical_and(np.logical_and(condition2, condition4), condition7):
                        chainage_temp.pop(k); dates_temp.pop(k); break
                    else:
                        condition4b = np.abs(diff_m2) > 1.5*cross_change
                        condition5b = np.abs(diff_p2) > 1.5*cross_change
                        condition8 = np.sign(diff_m2) == np.sign(diff_p2)
                        if np.logical_and(np.logical_and(np.logical_and(condition4b, condition5b),
                                                         np.logical_and(~condition1, ~condition2)), condition8):
                            chainage_temp.pop(k); dates_temp.pop(k); break
        k = k + 1
    return chainage_temp, dates_temp


# ---- MNDWI thresholds from the CoastSat output (for step 2) ----
mndwi = None
if not np.isnan(OTSU_THRESHOLD[0]):
    if os.path.exists(OUTPUT_PKL):
        import pickle
        o = pickle.load(open(OUTPUT_PKL, 'rb'))
        m = pd.Series(o['MNDWI_threshold'], index=pd.to_datetime(o['dates'], utc=True).tz_localize(None))
        m = m[~m.index.duplicated()]
        pos = m.index.get_indexer(dates, method='nearest', tolerance=pd.Timedelta('1min'))
        mndwi = pd.Series(np.where(pos >= 0, m.values[pos.clip(0)], np.nan), index=dates)
        print(f'MNDWI thresholds found for {mndwi.notna().sum()} of {len(dates)} images')
    else:
        print(f'{OUTPUT_PKL} not found -> skipping the MNDWI step (step 2)')


def despike(ts, name):
    """CoastSat reject_outliers steps 1-3, with every removal logged."""
    out, log = ts.copy(), []
    for t in transects:
        s1 = ts[t].dropna()                                              # 1. NaNs
        if s1.empty:
            continue
        if mndwi is None:                                                # 2. MNDWI threshold
            s2 = s1
        else:
            th = mndwi.reindex(s1.index)
            s2 = s1[(th >= OTSU_THRESHOLD[0]) & (th <= OTSU_THRESHOLD[1])]
            log += [(name, t, d, 'otsu', s1[d]) for d in s1.index.difference(s2.index)]
            if len(s2) < 30:                                             # CoastSat skips the transect
                log += [(name, t, d, 'transect dropped (<30 points)', s2[d]) for d in s2.index]
                out[t] = np.nan
                continue
        if len(s2) >= 3:                                                 # 3. despiking
            _, d3 = identify_outliers(list(s2.values), list(s2.index), MAX_CROSS_CHANGE)
        else:
            d3 = list(s2.index)
        kept = pd.DatetimeIndex(d3)
        log += [(name, t, d, 'spike', s2[d]) for d in s2.index.difference(kept)]
        out.loc[~out.index.isin(kept), t] = np.nan
    return out, log


def trim_gaps(ts, name):
    """Start each transect after its last gap longer than MAX_GAP."""
    out, log, starts = ts.copy(), [], {}
    for t in transects:
        d = ts[t].dropna().index
        if len(d) == 0:
            starts[t] = pd.NaT; continue
        gaps = d.to_series().diff()
        big = gaps[gaps > pd.Timedelta(MAX_GAP)]
        starts[t] = big.index[-1] if len(big) else d[0]
    if COMMON_START:
        s0 = max(v for v in starts.values() if pd.notna(v))
        starts = {t: s0 for t in starts}
    for t, s0 in starts.items():
        if pd.isna(s0):
            continue
        cut = ts.index[(ts.index < s0) & ts[t].notna()]
        log += [(name, t, d, 'before gap trim', ts.at[d, t]) for d in cut]
        out.loc[out.index < s0, t] = np.nan
    return out, log, starts

In [ ]:
versions = {'none': ts_none, 'tide': ts_tide, 'tide_sla': ts_tide_sla,
            'tide_sla_wave': ts_tide_sla_wave, 'tide_sla_wave_thr': ts_tide_sla_wave_thr}

clean, logs, starts_all = {}, [], {}
for name, ts in versions.items():
    d, log1 = despike(ts, name)
    c, log2, starts = trim_gaps(d, name)
    clean[name], starts_all[name] = c, starts
    logs += log1 + log2
    save_ts(c, f'{name}_despiked', transects)

removed = pd.DataFrame(logs, columns=['version', 'transect', 'date', 'reason', 'chainage'])
removed.to_csv(os.path.join(COASTSAT, 'removed_points_log.csv'), index=False)

### Summary of what was removed and when

In [ ]:
# points per version and reason
n_in = {k: int(v[transects].notna().sum().sum()) for k, v in versions.items()}
n_out = {k: int(v[transects].notna().sum().sum()) for k, v in clean.items()}
table = removed.pivot_table(index='version', columns='reason', values='chainage', aggfunc='count', fill_value=0)
table.insert(0, 'points in', pd.Series(n_in))
table['points kept'] = pd.Series(n_out)
print(table.loc[list(versions)].to_string(), '\n')

# start date of each transect after trimming
print('series start after gap trimming:')
print(pd.DataFrame(starts_all).apply(lambda c: c.dt.strftime('%Y-%m-%d')).to_string(), '\n')

# when the spikes were removed: count per year and version
spikes = removed[removed.reason == 'spike']
print('spikes removed per year:')
print(spikes.pivot_table(index=spikes.date.dt.year, columns='version', values='chainage',
                         aggfunc='count', fill_value=0).reindex(columns=list(versions), fill_value=0).to_string())

In [ ]:
# QA figure for one transect and version
T, V = transects[len(transects) // 2], 'tide_sla'
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(versions[V].index, versions[V][T], '.', color='0.7', label='corrected, before cleaning')
r = removed[(removed.version == V) & (removed.transect == T)]
for reason, col in [('otsu', 'C1'), ('spike', 'C3'), ('before gap trim', 'C0')]:
    rr = r[r.reason == reason]
    ax.plot(rr.date, rr.chainage, 'o', mfc='none', color=col, label=f'{reason} ({len(rr)})')
ax.plot(clean[V].index, clean[V][T], '-', color='k', lw=0.8, label='final')
ax.set(ylabel='chainage (m)', title=f'{V}, transect {T}')
ax.legend(fontsize=8, ncol=5)

## Validation against drone footage